# Лекция: Бинарная (логистическая) регрессия в Python

**Дисциплина:** Введение в анализ больших данных

Когда отклик принимает **только два значения** (0/1), используют логистическую регрессию:

$$
P(Y=1 \mid x) = \frac{1}{1 + \exp(-(b_0 + b_1 x_1 + \ldots + b_k x_k))}
$$

Инструменты: `statsmodels` (`logit` / `probit`), метрики и ROC из **scikit-learn**.

Демо: датасет **breast cancer** (диагностика опухоли). Примеры **не совпадают** с лабораторным заданием — его выполните самостоятельно.


## 0. Импорт


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, accuracy_score, classification_report,
    roc_curve, roc_auc_score, ConfusionMatrixDisplay,
)

plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")
np.random.seed(42)
print("Библиотеки загружены")


---
## 1. Данные

Отклик `target`: 0 — злокачественная, 1 — доброкачественная (как в sklearn).  
Предикторы — несколько числовых характеристик опухоли.


In [ ]:
raw = load_breast_cancer(as_frame=True)
df = raw.frame.copy()
df = df.rename(columns={
    "mean radius": "radius",
    "mean texture": "texture",
    "mean perimeter": "perimeter",
    "mean area": "area",
    "mean smoothness": "smoothness",
})
use = ["target", "radius", "texture", "perimeter", "area", "smoothness"]
df = df[use]
print(df.head())
print("\ntarget:")
print(df["target"].value_counts())


---
## 2. Логит-модель

`smf.logit("y ~ x1 + x2 + ...", data=df).fit()`


In [ ]:
m_logit = smf.logit(
    "target ~ radius + texture + smoothness",
    data=df,
).fit(disp=False)
print(m_logit.summary())


### Пробит-модель

Тот же набор предикторов, другая link-функция.


In [ ]:
m_probit = smf.probit(
    "target ~ radius + texture + smoothness",
    data=df,
).fit(disp=False)
print(m_probit.summary().tables[1])


### Значимость модели в целом (likelihood-ratio)


In [ ]:
lr = 2 * (m_logit.llf - m_logit.llnull)
p_lr = stats.chi2.sf(lr, m_logit.df_model)
print(f"LR chi2 = {lr:.2f}, df = {m_logit.df_model:.0f}, p = {p_lr:.4e}")
print("модель значима" if p_lr < 0.05 else "модель не значима")


---
## 3. Матрица неточностей

Порог по умолчанию: 0.5.

| | прогноз 0 | прогноз 1 |
|--|-----------|-----------|
| факт 0 | TN | FP |
| факт 1 | FN | TP |

- **Accuracy** = (TP+TN)/N  
- **Sensitivity (Recall)** = TP/(TP+FN)  
- **Specificity** = TN/(TN+FP)


In [ ]:
prob = m_logit.predict(df)
pred = (prob >= 0.5).astype(int)

cm = confusion_matrix(df["target"], pred)
print("Confusion matrix:\n", cm)
print("Accuracy =", round(accuracy_score(df["target"], pred), 4))
print(classification_report(df["target"], pred, digits=3))

ConfusionMatrixDisplay(cm, display_labels=[0, 1]).plot(cmap="Blues")
plt.title("Матрица неточностей (logit, thr=0.5)")
plt.show()


---
## 4. ROC-кривая и AUC

Чем ближе кривая к верхнему левому углу и чем больше **AUC**, тем лучше.  
AUC ≈ 0.5 — как случайное угадывание; ближе к 1 — сильная модель.


In [ ]:
fpr, tpr, _ = roc_curve(df["target"], prob)
auc = roc_auc_score(df["target"], prob)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, lw=2, label=f"AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], "k--", label="случайный")
plt.xlabel("1 − Specificity (FPR)")
plt.ylabel("Sensitivity (TPR)")
plt.title("ROC (logit)")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"AUC = {auc:.4f}")


---
## 5. Train / test

Правильный порядок: обучить на train, оценить на **test** (и при желании на train для сравнения).


In [ ]:
train, test = train_test_split(
    df, test_size=0.3, random_state=7, stratify=df["target"]
)
print("train:", train.shape, "test:", test.shape)
print("доля target=1 train:", round(train["target"].mean(), 3))
print("доля target=1 test :", round(test["target"].mean(), 3))


In [ ]:
model = smf.logit(
    "target ~ radius + texture + smoothness",
    data=train,
).fit(disp=False)
print(model.summary().tables[1])


In [ ]:
def evaluate(model, data, name):
    y = data["target"]
    p = model.predict(data)
    yhat = (p >= 0.5).astype(int)
    acc = accuracy_score(y, yhat)
    auc = roc_auc_score(y, p)
    cm = confusion_matrix(y, yhat)
    print(f"=== {name} ===")
    print("CM:\n", cm)
    print(f"Accuracy = {acc:.4f}, AUC = {auc:.4f}")
    print(classification_report(y, yhat, digits=3))
    return p, auc

p_tr, auc_tr = evaluate(model, train, "TRAIN")
p_te, auc_te = evaluate(model, test, "TEST")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, data, p, title, auc_v in [
    (axes[0], train, p_tr, "Train", auc_tr),
    (axes[1], test, p_te, "Test", auc_te),
]:
    fpr, tpr, _ = roc_curve(data["target"], p)
    ax.plot(fpr, tpr, lw=2, label=f"AUC={auc_v:.3f}")
    ax.plot([0, 1], [0, 1], "k--")
    ax.set_title(f"ROC ({title})")
    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR")
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Как описать результаты

1. Значимость коэффициентов (p < 0.05) и знак связи.  
2. Метрики на **train** и **test** (accuracy, sensitivity, specificity, AUC).  
3. Нет ли сильного проседания на test (переобучение).  
4. Форма ROC и величина AUC (ориентиры: >0.7 приемлемо, >0.8 хорошо).

---
## Шпаргалка по методам (Python)

| Задача | Код |
|--------|-----|
| Логит | `smf.logit("y ~ x1 + x2", data=df).fit()` |
| Пробит | `smf.probit("y ~ x1 + x2", data=df).fit()` |
| Вероятности | `model.predict(df)` |
| Класс (порог 0.5) | `(prob >= 0.5).astype(int)` |
| Confusion matrix | `confusion_matrix(y, pred)` |
| Accuracy | `accuracy_score(y, pred)` |
| ROC / AUC | `roc_curve`, `roc_auc_score` |
| Train/test | `train_test_split(..., stratify=y)` |

---
## Что сделать после лекции

1. Повторите logit/probit на **других** предикторах.  
2. Откройте лабораторное задание (свой файл, train/test) и выполните **самостоятельно**.  
3. Порог 0.5 можно менять, опираясь на ROC.

Удачи!
